# Retail Sales — Exploratory Data Analysis

**Dataset:** Synthetic e-commerce order data (structured to mirror the [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)). Swap in the real Olist CSVs and this notebook runs unchanged — see `src/generate_data.py` for the column mapping.

**Goal:** Understand overall sales performance, seasonality, and where revenue is concentrated (by category and region) before drilling into any specific question.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
%matplotlib inline

## 1. Load and inspect the raw data

In [ ]:
df_raw = pd.read_csv("../data/orders.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"
])
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

## 2. Data quality check — missing values and duplicates

Always do this before trusting any downstream numbers.

In [ ]:
missing = df_raw.isnull().mean().sort_values(ascending=False)
missing = missing[missing > 0] * 100
print(missing)

fig, ax = plt.subplots(figsize=(7, 3))
sns.barplot(x=missing.values, y=missing.index, ax=ax, color="#4C72B0")
ax.set_xlabel("% missing")
ax.set_title("Missing Values by Column")
plt.tight_layout()
plt.show()

In [ ]:
n_dupes = df_raw.duplicated(subset="order_id").sum()
print(f"Duplicate order_ids: {n_dupes}")

**Finding:** `review_score` has a small amount of missingness (expected — not every customer leaves a review) and there are a handful of duplicate `order_id`s to drop. Both are handled in the cleaning step below.

## 3. Clean the data

In [ ]:
df = df_raw.drop_duplicates(subset="order_id").copy()
df["revenue"] = df["price"] + df["freight_value"]
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
df["order_weekday"] = df["order_purchase_timestamp"].dt.day_name()

print("Cleaned shape:", df.shape)
df[["order_id", "revenue", "order_month", "order_weekday"]].head()

## 4. Revenue trend over time

In [ ]:
monthly = df.groupby("order_month")["revenue"].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(monthly["order_month"], monthly["revenue"], marker="o", color="#4C72B0")
ax.set_title("Monthly Revenue Trend")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

**Finding:** Clear seasonality — revenue peaks in November/December (holiday shopping) and dips in February. Worth planning inventory and staffing around this.

## 5. Seasonality — day of week and month

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
by_weekday = df.groupby("order_weekday")["revenue"].sum().reindex(weekday_order)
sns.barplot(x=by_weekday.index, y=by_weekday.values, ax=axes[0], color="#55A868")
axes[0].set_title("Revenue by Day of Week")
axes[0].tick_params(axis="x", rotation=45)

df["month_num"] = df["order_purchase_timestamp"].dt.month
by_month = df.groupby("month_num")["revenue"].sum()
sns.barplot(x=by_month.index, y=by_month.values, ax=axes[1], color="#C44E52")
axes[1].set_title("Revenue by Month (all years combined)")

plt.tight_layout()
plt.show()

## 6. Top product categories

In [ ]:
top_cat = df.groupby("product_category")["revenue"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=top_cat.values, y=top_cat.index, ax=ax, color="#4C72B0")
ax.set_title("Revenue by Product Category")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.show()

top_cat.head(5)

**Finding:** Electronics, furniture, and computer accessories drive the most revenue — these are the categories worth prioritizing for ad spend and inventory investment.

## 7. Regional performance

In [ ]:
by_state = df.groupby("customer_state")["revenue"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=by_state.values, y=by_state.index, ax=ax, color="#8172B2")
ax.set_title("Revenue by Customer State")
ax.set_xlabel("Revenue")
plt.tight_layout()
plt.show()

## 8. Order value distribution — check for outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.histplot(df["revenue"], bins=50, ax=axes[0], color="#4C72B0")
axes[0].set_title("Order Value Distribution")

sns.boxplot(x=df["revenue"], ax=axes[1], color="#DD8452")
axes[1].set_title("Order Value — Outlier Check")

plt.tight_layout()
plt.show()

print(df["revenue"].describe())

## Summary of findings

- Revenue shows strong holiday-season seasonality (Nov/Dec peak, Feb trough)
- Electronics, furniture, and computer accessories are the top revenue categories
- Revenue is concentrated in a handful of states — SP, RJ, and MG together account for a large share
- Order values are right-skewed with a long tail of high-value orders, as expected for e-commerce

**Next:** see `02_deep_dive.ipynb` for a focused analysis of whether delivery delays are hurting customer satisfaction — a specific, actionable business question.